# Personalized Learning — A2A + MCP Workshop

> **No API key required.** Every agent reads from local JSON files via MCP. You can run this entire workshop — including all three agents and all tests — completely offline.

## What you will build

- **Assessment Agent** — an A2A agent that quizzes users on topics, evaluates their score, and tracks their learning level across multi-turn conversations

## What you will learn

- How MCP tools expose structured data to agents (no LLM calls needed)
- How A2A Agent Cards enable automatic Orchestrator routing
- How `context_id` and conversation history enable multi-turn memory
- How to test a multi-agent system bottom-up: tools → agents → orchestrator → memory

## Architecture

```
User
  └─> Orchestrator         (:8090)   [pre-built, LLM-backed keyword routing]
        ├─> Topic Explainer  (:8091)   [pre-built]  ─┐
        ├─> Assessment Agent (:8092)   [YOU BUILD]  ─┤─> MCP Server (:8004)  [100% offline]
        └─> Study Plan Agent (:8093)   [pre-built]  ─┘
```

## Workshop Flow

| Step | What you do |
|------|-------------|
| **Part 1** | Start the MCP Server — the shared data layer — and explore its tools |
| **Part 2** | Start 2 pre-built agents and inspect their Agent Cards |
| **Part 3** | **Build** the Assessment Agent yourself (3 logic flows) |
| **Part 4** | Run your Assessment Agent and smoke-test it |
| **Part 5** | Start the Orchestrator and observe keyword routing |
| **Part 6** | Run the full test suite |


In [ ]:
%%html
<div id="arch-diagram" class="mermaid" style="max-width:700px;margin:16px auto">
flowchart LR
    User(["User"])
    subgraph system ["A2A Multi-Agent System"]
        direction TB
        Orch["Orchestrator\n:8090\nLLM keyword routing"]
        TE["Topic Explainer\n:8091\nrule-based"]
        AA["Assessment Agent\n:8092\nYOU BUILD"]
        SP["Study Plan Agent\n:8093\nrule-based"]
    end
    MCP["MCP Server\n:8004\n8 tools · 100% offline"]

    User --> Orch
    Orch --> TE
    Orch --> AA
    Orch --> SP
    TE --> MCP
    AA --> MCP
    SP --> MCP
</div>
<script>
  (function () {
    var s = document.createElement("script");
    s.src = "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js";
    s.onload = function () {
      mermaid.initialize({ startOnLoad: false, theme: "neutral" });
      mermaid.run();
    };
    document.head.appendChild(s);
  })();
</script>

---
## Part 0 — Setup


In [ ]:
# ── Step 1: Get the code ────────────────────────────────────────────────────
# Choose ONE option:
#   Option A: clone from GitHub  →  update GITHUB_REPO below, keep USE_DRIVE = False
#   Option B: mount Google Drive →  set USE_DRIVE = True

GITHUB_REPO = 'https://github.com/YOUR_ORG/a2a_mcp_workshop.git'  # <-- UPDATE before running
USE_DRIVE   = True  # set True to use Google Drive instead

import os, subprocess, sys

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/A2A_workshop/personalized_learning'  # update if needed
else:
    if not os.path.exists('/content/a2a_mcp_workshop'):
        subprocess.run(['git', 'clone', GITHUB_REPO], check=True)
    REPO_PATH = '/content/a2a_mcp_workshop'

print(f'Repo path: {REPO_PATH}')


In [ ]:
# ── Step 2: Install dependencies and set working directory ──────────────────
LEARNING_DIR = os.path.join(REPO_PATH)
os.chdir(LEARNING_DIR)
print(f'Working directory: {os.getcwd()}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests', 'pytest', 'pytest-asyncio', 'fastmcp', 'python-dotenv', 'a2a-sdk==0.3.22', 'langchain-openai', 'langchain-core', 'streamlit', 'httpx', 'uvicorn', 'starlette'], check=True)

os.environ['MCP_SERVER_URL'] = 'http://127.0.0.1:8004/mcp'
print('Done. MCP_SERVER_URL =', os.environ['MCP_SERVER_URL'])


In [ ]:
# ── Step 3: Shared utilities (run this cell once, then leave it) ─────────────
import subprocess, sys, os, time, requests

_services = {}

def start_service(name, module, port):
    """Launch a Python module as a background subprocess."""
    env = {
        **os.environ,
        'MCP_SERVER_URL': 'http://127.0.0.1:8004/mcp',
        'PYTHONPATH': os.getcwd(),
    }
    log_path = f'/tmp/{name.lower().replace(" ", "_")}.log'
    log_file = open(log_path, 'w')
    proc = subprocess.Popen(
        [sys.executable, '-m', module],
        stdout=log_file,
        stderr=log_file,
        env=env,
        cwd=os.getcwd(),
    )
    _services[name] = (proc, log_file, log_path)
    print(f'  Started {name}  pid={proc.pid}  port={port}')
    return proc


def wait_for_service(url, name='service', timeout=30):
    """Poll url every second until it returns a non-5xx status (or timeout)."""
    for i in range(timeout):
        try:
            r = requests.get(url, timeout=2)
            if r.status_code < 500:
                print(f'  [OK] {name} is ready')
                return True
        except Exception:
            pass
        time.sleep(1)
        if (i + 1) % 5 == 0:
            print(f'  ... waiting for {name} ({i + 1}s elapsed)')
    print(f'  [FAIL] {name} did not respond within {timeout}s')
    return False


def show_log(name, lines=30):
    """Print the last N lines from a service log (useful for debugging)."""
    _, _, log_path = _services.get(name, (None, None, f'/tmp/{name.lower().replace(" ", "_")}.log'))
    try:
        with open(log_path) as f:
            content = f.readlines()
        print(f'--- {name} log (last {lines} lines) ---')
        print(''.join(content[-lines:]))
    except FileNotFoundError:
        print(f'No log file found for {name}')


def kill_all_services():
    """Terminate all background services started in this session."""
    for name, (proc, log_file, _) in _services.items():
        proc.terminate()
        log_file.close()
        print(f'  Stopped {name}')
    _services.clear()


print('Utilities loaded.')


---
## Part 1 — MCP Server

> **Why start here?** The MCP Server is the shared data layer that all three remote agents depend on. Starting it first ensures agents can reach their tools as soon as they boot. Think of it as the database — it must be up before the application servers.

The **MCP Server** (port 8004) exposes 8 tools over the MCP HTTP protocol. All data comes from **local JSON files** in `mcp/data/` — no external API calls, no credentials required.

| Tool | What it does |
|------|--------------|
| `get_topic_summary` | Explanation of a topic at a given level |
| `get_assessment_questions_by_topic` | Quiz questions per topic + level |
| `get_learning_state` | User's current level for a topic |
| `update_learning_state` | Level-up if score ≥ 75% |
| `get_study_path` | Ordered study steps by topic/level/time |
| `get_job_description` | Job requirements |
| `get_resume_profile` | Candidate skills |
| `get_skill_gap_analysis` | Missing skills + recommended topics |

**Supported topics:** `mcp`, `a2a`, `rag`, `prompt_engineering`, `python_async`  
**Supported levels:** `beginner`, `intermediate`, `advanced`


In [ ]:
print('Starting MCP Server...')
start_service('MCP Server', 'mcp.fastmcp_server', 8004)
time.sleep(3)
wait_for_service('http://127.0.0.1:8004/', 'MCP Server', timeout=20)


In [ ]:
# Quick demo: call get_topic_summary to confirm MCP is working
import asyncio, json
from fastmcp import Client

async def demo_mcp():
    async with Client('http://127.0.0.1:8004/mcp') as client:
        result = await client.call_tool('get_topic_summary', {'topic': 'mcp', 'level': 'beginner'})
        data = json.loads(result.content[0].text)
        print('--- MCP Tool: get_topic_summary ---')
        print(f"Topic    : {data['topic']}  |  Level : {data['level']}")
        print(f"Summary  : {data['summary'][:160]}...")
        print(f"Concepts : {data['key_concepts']}")
        print(f"Source   : {data['data_source']}")  # "local_json" confirms 100% offline operation

await demo_mcp()


In [ ]:
# Raw MCP response: see the full JSON structure returned by get_assessment_questions_by_topic
import json
from fastmcp import Client

async def show_raw_mcp_response():
    async with Client('http://127.0.0.1:8004/mcp') as client:
        result = await client.call_tool(
            'get_assessment_questions_by_topic',
            {'topic': 'mcp', 'level': 'beginner', 'limit': 2}
        )
        data = json.loads(result.content[0].text)

    print('=== Raw MCP Tool Response: get_assessment_questions_by_topic ===')
    print(json.dumps(data, indent=2))
    print()
    print('--- Field guide ---')
    print(f"  topic           : {data['topic']!r}   ← the queried topic")
    print(f"  level           : {data['level']!r}   ← beginner / intermediate / advanced")
    print(f"  questions       : {len(data['questions'])} items  ← the actual quiz payload")
    print(f"  total_available : {data['total_available']}       ← questions available in JSON for this level")
    print(f"  data_source     : {data['data_source']!r}  ← always 'local_json' (no external API calls)")
    print()
    print('  Each question has: question, expected_answer, explanation, level, score_weight')
    if data['questions']:
        print(f"  Example: {data['questions'][0]['question']!r}")

await show_raw_mcp_response()

---
### Part 1 Checkpoint

After running the cells above you should have seen:
- `[OK] MCP Server is ready` on port 8004
- A `get_topic_summary` response with `"data_source": "local_json"`
- A `get_assessment_questions_by_topic` response with 2 quiz questions and a `total_available` count

**If the MCP server did not start**, run `show_log('MCP Server')` in a new cell to see the error.

---

---
## Part 2 — Pre-built Remote Agents

Two of the three remote agents come pre-built. We start them here so the Orchestrator can route to them later.

> **What to observe:** Each agent publishes an **Agent Card** at `/.well-known/agent-card.json`. The card contains `skills`, `tags`, and `examples` that the Orchestrator uses to build its keyword routing index. After starting the agents, we will inspect their cards — then in Part 5 you will see exactly how those tags determine which agent handles each message.

### Topic Explainer Agent (port 8091)
Explains topics (MCP, A2A, RAG, prompt_engineering, python_async) at a requested level.  
MCP tool used: `get_topic_summary`

### Study Plan Agent (port 8093)
Builds ordered learning plans based on topic, level, and available time. Also handles career plans using skill gap analysis.  
MCP tools used: `get_learning_state`, `get_study_path`, `get_skill_gap_analysis`


In [ ]:
print('Starting pre-built remote agents...')
start_service('Topic Explainer', 'a2a_agents.remote_agents.topic_explainer_agent', 8091)
start_service('Study Plan Agent', 'a2a_agents.remote_agents.study_plan_agent', 8093)

time.sleep(4)
wait_for_service('http://localhost:8091/.well-known/agent-card.json', 'Topic Explainer')
wait_for_service('http://localhost:8093/.well-known/agent-card.json', 'Study Plan Agent')


In [ ]:
# Raw agent response: call Topic Explainer directly (bypassing the Orchestrator)
from tests.helpers import call_agent

print('=== Direct Agent Call: Topic Explainer (port 8091) ===')
print('Request: "Explain MCP for a beginner"\n')

resp = await call_agent('http://localhost:8091', 'Explain MCP for a beginner')

print('--- Raw agent response ---')
print(resp)

print()
print('--- Response structure ---')
lines = [ln for ln in resp.strip().split('\n') if ln.strip()]
print(f'  Total lines      : {len(lines)}')
print(f'  First line       : {lines[0]!r}')
print(f'  Has key concepts : {"concept" in resp.lower() or "concept" in resp.lower()}')
print()
print('Note: In Part 5 the Orchestrator returns this same content.')
print('      Routing is transparent — the agent response is never modified.')

In [ ]:
# Inspect the agent cards — these tags drive the Orchestrator's routing decisions
import requests, json

for name, port in [('Topic Explainer', 8091), ('Study Plan Agent', 8093)]:
    card = requests.get(f'http://localhost:{port}/.well-known/agent-card.json').json()
    print(f'\n=== {name} (port {port}) ===')
    print(f'  Name       : {card["name"]}')
    print(f'  Description: {card["description"][:100]}...')
    for skill in card.get('skills', []):
        print(f'  Skill      : {skill["name"]}')
        print(f'  Tags       : {skill.get("tags", [])}')
        print(f'  Examples   : {skill.get("examples", [])[:3]}')
    print(f'  ^ These tags and examples are indexed by the Orchestrator at startup.')
    print(f'    Messages matching these keywords will be routed to {name}.')


---
### Part 2 Checkpoint

After running the cells above you should have:
- `[OK] Topic Explainer is ready` on port 8091
- `[OK] Study Plan Agent is ready` on port 8093
- Seen a raw agent response (a structured topic explanation)
- Seen the agent card tags that will drive Orchestrator routing in Part 5

**If an agent did not start**, run `show_log('Topic Explainer')` or `show_log('Study Plan Agent')` in a new cell.

---

---
## Part 3 — Exercise: Build the Assessment Agent

The **Assessment Agent** (port 8092) handles three conversation flows. Each flow detects a different user intent and calls a different MCP tool.

---

### Your task: implement three flows in `agent_logic.py`

| Flow | User says | MCP tool | Returns |
|------|-----------|----------|---------|
| **1 — Generate Quiz** (default) | *"Quiz me on A2A"* | `get_learning_state` + `get_assessment_questions_by_topic` | numbered question list |
| **2 — Score Reporting** | *"I got 3 out of 4 correct"* | `update_learning_state` | previous level, new level, message |
| **3 — Level Query** | *"What is my current MCP level?"* | `get_learning_state` | current_level |

### Steps

1. Read the flow diagram below — understand what each branch does
2. In the **agent_logic** cell, find the three `# TODO` blocks and implement each flow using the MCP tool reference in the next cell
3. Run the **agent_executor** and **\_\_main\_\_** cells as-is (no changes needed)
4. Proceed to Part 4 to start and test your agent

> **Hint — Flow 1 score detection:** `_parse_score(user_input)` returns `(correct_count, total_count)` for phrases like *"3 out of 4"*, *"3/4"*, *"3 from 4"*, or `None` if no score is found. Check for this first before checking intent keywords.

---

### Flow details

**Flow 1 — Score Reporting** (triggered when `_parse_score` returns a result)
```
_parse_score(user_input)  →  (correct_count=3, total_count=4)
update_learning_state(user_id='user_1', topic, correct_count=3, total_count=4)
  → {previous_level, new_level, score, message}
yield content: "Your {topic} level update: ..."
```

**Flow 2 — Level Query** (triggered by "what is my level" or "current level" in input)
```
get_learning_state(user_id='user_1', topic)
  → {current_level: 'beginner' | 'intermediate' | 'advanced', ...}
yield content: "Your current {topic} level is {level}."
```

**Flow 3 — Generate Quiz** (default — reached when neither Flow 1 nor Flow 2 triggered)
```
get_learning_state(user_id='user_1', topic)           → {current_level}
get_assessment_questions_by_topic(topic, level, limit=4)
  → {questions: [{question, expected_answer, ...}, ...]}
yield content: "Assessment questions for {topic} ({level} level):\nQuestion 1: ..."
```


In [ ]:
%%html
<div id="flows-diagram" class="mermaid" style="max-width:680px;margin:16px auto">
flowchart TD
    Start(["user_input"]) --> ParseScore{"_parse_score\nfinds a number?"}
    ParseScore -->|"Yes — e.g. '3 out of 4'"| Flow1["Flow 1: Score Reporting\ncall update_learning_state\nyield level update message"]
    ParseScore -->|"No"| CheckLevel{"'what is my level'\nor 'current level'\nin input?"}
    CheckLevel -->|"Yes"| Flow2["Flow 2: Level Query\ncall get_learning_state\nyield current level string"]
    CheckLevel -->|"No"| Flow3["Flow 3: Generate Quiz  (default)\ncall get_learning_state → current level\ncall get_assessment_questions\nyield numbered question list"]
</div>
<script>
  (function () {
    var s = document.createElement("script");
    s.src = "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js";
    s.onload = function () {
      mermaid.initialize({ startOnLoad: false, theme: "neutral" });
      mermaid.run();
    };
    document.head.appendChild(s);
  })();
</script>

In [ ]:
%%writefile mcp/fastmcp_server.py
import json
import logging
import os
from pathlib import Path
from copy import deepcopy

from dotenv import load_dotenv
from fastmcp import FastMCP

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s [%(name)s] %(message)s',
    handlers=[logging.StreamHandler()],
)

logger = logging.getLogger('mcp.learning.tools')

mcp = FastMCP()

# ---------------------------------------------------------------------------
# Load all static data files at startup
# ---------------------------------------------------------------------------
_DATA_DIR = Path(__file__).parent / 'data'

with open(_DATA_DIR / 'learning' / 'topics.json', encoding='utf-8') as _f:
    TOPICS: dict = json.load(_f)

with open(_DATA_DIR / 'learning' / 'assessment_questions.json', encoding='utf-8') as _f:
    ASSESSMENT_QUESTIONS: dict = json.load(_f)

with open(_DATA_DIR / 'learning' / 'study_paths.json', encoding='utf-8') as _f:
    STUDY_PATHS: dict = json.load(_f)

with open(_DATA_DIR / 'learning' / 'user_learning_state.json', encoding='utf-8') as _f:
    _SEED_STATE: dict = json.load(_f)

with open(_DATA_DIR / 'career' / 'job_descriptions.json', encoding='utf-8') as _f:
    JOB_DESCRIPTIONS: dict = json.load(_f)

with open(_DATA_DIR / 'career' / 'sample_resumes.json', encoding='utf-8') as _f:
    SAMPLE_RESUMES: dict = json.load(_f)

with open(_DATA_DIR / 'career' / 'skill_map.json', encoding='utf-8') as _f:
    SKILL_MAP: dict = json.load(_f)

# In-memory learning state (seeded from JSON, updated by update_learning_state)
_LEARNING_STATE: dict = deepcopy(_SEED_STATE)

VALID_TOPICS  = list(TOPICS.keys())
VALID_LEVELS  = ['beginner', 'intermediate', 'advanced']
VALID_TIMES   = ['30_minutes', '2_hours', '1_day']
LEVEL_ORDER   = {'beginner': 0, 'intermediate': 1, 'advanced': 2}
LEVEL_UP      = {0: 'intermediate', 1: 'advanced', 2: 'advanced'}


# ---------------------------------------------------------------------------
# Tools
# ---------------------------------------------------------------------------

@mcp.tool(tags={'topic', 'explanation'})
def get_topic_summary(topic: str, level: str = 'beginner') -> dict:
    """
    Get a structured summary of a learning topic at a specific level.

    Args:
        topic: One of: mcp, a2a, rag, prompt_engineering, python_async
        level: One of: beginner, intermediate, advanced

    Returns:
        Dict with topic, level, summary, key_concepts, common_misconceptions,
        next_step suggestion, and data_source.
    """
    topic_key = topic.lower().strip().replace(' ', '_')
    level_key = level.lower().strip()

    if topic_key not in TOPICS:
        return {
            'topic': topic,
            'level': level,
            'found': False,
            'message': f"Topic '{topic}' not found. Available topics: {', '.join(VALID_TOPICS)}.",
            'data_source': 'local_json',
        }

    if level_key not in VALID_LEVELS:
        level_key = 'beginner'

    data = TOPICS[topic_key][level_key]

    return {
        'topic': topic_key,
        'level': level_key,
        'found': True,
        'summary': data['summary'],
        'key_concepts': data['key_concepts'],
        'common_misconceptions': data['common_misconceptions'],
        'next_step': data.get('next_step', ''),
        'data_source': 'local_json',
    }


@mcp.tool(tags={'assessment', 'quiz'})
def get_assessment_questions_by_topic(
    topic: str,
    level: str = 'beginner',
    limit: int = 4,
) -> dict:
    """
    Get assessment questions for a topic at a given level.

    Args:
        topic: One of: mcp, a2a, rag, prompt_engineering, python_async
        level: One of: beginner, intermediate, advanced
        limit: Maximum number of questions to return (default 4)

    Returns:
        Dict with topic, level, questions list, and data_source.
        Each question has: id, question, expected_answer, explanation, level, score_weight.
    """
    topic_key = topic.lower().strip().replace(' ', '_')
    level_key = level.lower().strip()

    if topic_key not in ASSESSMENT_QUESTIONS:
        return {
            'topic': topic,
            'level': level,
            'found': False,
            'questions': [],
            'message': f"Topic '{topic}' not found. Available topics: {', '.join(VALID_TOPICS)}.",
            'data_source': 'local_json',
        }

    if level_key not in VALID_LEVELS:
        level_key = 'beginner'

    questions = ASSESSMENT_QUESTIONS[topic_key].get(level_key, [])
    limited = questions[:max(1, min(limit, len(questions)))]

    return {
        'topic': topic_key,
        'level': level_key,
        'found': True,
        'questions': limited,
        'total_available': len(questions),
        'data_source': 'local_json',
    }


@mcp.tool(tags={'assessment', 'state'})
def get_learning_state(user_id: str, topic: str) -> dict:
    """
    Get the current learning state for a user on a specific topic.

    Args:
        user_id: User identifier (e.g. "user_1")
        topic: One of: mcp, a2a, rag, prompt_engineering, python_async

    Returns:
        Dict with current_level, answered_questions, correct_answers,
        total_answers, confidence_score, and data_source.
    """
    topic_key = topic.lower().strip().replace(' ', '_')

    user_state = _LEARNING_STATE.get(user_id, {})
    topic_state = user_state.get(topic_key)

    if topic_state is None:
        return {
            'user_id': user_id,
            'topic': topic_key,
            'current_level': 'beginner',
            'answered_questions': [],
            'correct_answers': 0,
            'total_answers': 0,
            'confidence_score': 0.0,
            'note': 'No prior state found. Starting at beginner level.',
            'data_source': 'local_json',
        }

    return {
        'user_id': user_id,
        'topic': topic_key,
        'current_level': topic_state['current_level'],
        'answered_questions': topic_state.get('answered_questions', []),
        'correct_answers': topic_state.get('correct_answers', 0),
        'total_answers': topic_state.get('total_answers', 0),
        'confidence_score': topic_state.get('confidence_score', 0.0),
        'data_source': 'local_json',
    }


@mcp.tool(tags={'assessment', 'state'})
def update_learning_state(
    user_id: str,
    topic: str,
    correct_count: int,
    total_count: int,
) -> dict:
    """
    Update a user's learning state based on quiz results.

    Level-up logic:
    - score = correct_count / total_count
    - if score >= 0.75: advance one level (beginner -> intermediate -> advanced)
    - otherwise: stay at current level
    - advanced stays advanced

    Args:
        user_id: User identifier
        topic: One of: mcp, a2a, rag, prompt_engineering, python_async
        correct_count: Number of questions answered correctly
        total_count: Total number of questions attempted

    Returns:
        Dict with previous_level, new_level, score, message, and data_source.
    """
    if total_count <= 0:
        return {
            'user_id': user_id,
            'topic': topic,
            'previous_level': 'unknown',
            'new_level': 'unknown',
            'score': 0.0,
            'message': 'total_count must be greater than 0.',
            'data_source': 'local_json',
        }

    topic_key = topic.lower().strip().replace(' ', '_')
    score = correct_count / total_count

    if user_id not in _LEARNING_STATE:
        _LEARNING_STATE[user_id] = {}
    if topic_key not in _LEARNING_STATE[user_id]:
        _LEARNING_STATE[user_id][topic_key] = {
            'current_level': 'beginner',
            'answered_questions': [],
            'correct_answers': 0,
            'total_answers': 0,
            'confidence_score': 0.0,
        }

    state = _LEARNING_STATE[user_id][topic_key]
    previous_level = state['current_level']
    previous_idx   = LEVEL_ORDER.get(previous_level, 0)

    if score >= 0.75:
        new_level = LEVEL_UP[previous_idx]
        message = (
            f'Excellent! Score {score:.0%}. '
            f'Level advanced from {previous_level} to {new_level}.'
            if new_level != previous_level
            else f'Outstanding! Score {score:.0%}. Already at maximum level: {new_level}.'
        )
    else:
        new_level = previous_level
        message = (
            f'Score {score:.0%}. Staying at {new_level} level. '
            f'Aim for 75% or higher to advance.'
        )

    state['current_level']    = new_level
    state['correct_answers']  = state.get('correct_answers', 0) + correct_count
    state['total_answers']    = state.get('total_answers', 0) + total_count
    total_so_far = state['total_answers']
    state['confidence_score'] = round(state['correct_answers'] / total_so_far, 2) if total_so_far else 0.0

    return {
        'user_id': user_id,
        'topic': topic_key,
        'previous_level': previous_level,
        'new_level': new_level,
        'score': round(score, 2),
        'message': message,
        'data_source': 'local_json',
    }


@mcp.tool(tags={'study', 'plan'})
def get_study_path(
    topic: str,
    level: str = 'beginner',
    available_time: str = '2_hours',
) -> dict:
    """
    Get a personalized study path for a topic, level, and available time.

    Args:
        topic: One of: mcp, a2a, rag, prompt_engineering, python_async
        level: One of: beginner, intermediate, advanced
        available_time: One of: 30_minutes, 2_hours, 1_day

    Returns:
        Dict with learning_objectives, ordered_steps, practice_suggestions,
        estimated_total_time, and data_source.
    """
    topic_key = topic.lower().strip().replace(' ', '_')
    level_key = level.lower().strip()
    time_key  = available_time.lower().strip().replace(' ', '_')

    if topic_key not in STUDY_PATHS:
        return {
            'topic': topic,
            'level': level,
            'available_time': available_time,
            'found': False,
            'message': f"Topic '{topic}' not found. Available topics: {', '.join(VALID_TOPICS)}.",
            'data_source': 'local_json',
        }

    if level_key not in VALID_LEVELS:
        level_key = 'beginner'
    if time_key not in VALID_TIMES:
        time_key = '2_hours'

    path = STUDY_PATHS[topic_key][level_key][time_key]

    return {
        'topic': topic_key,
        'level': level_key,
        'available_time': time_key,
        'found': True,
        'learning_objectives': path['learning_objectives'],
        'ordered_steps': path['ordered_steps'],
        'practice_suggestions': path['practice_suggestions'],
        'estimated_total_time': path['estimated_total_time'],
        'data_source': 'local_json',
    }


@mcp.tool(tags={'career', 'job'})
def get_job_description(job_id: str) -> dict:
    """
    Get a job description by job ID.

    Args:
        job_id: One of: ai_engineer, data_scientist, backend_engineer

    Returns:
        Dict with job_id, title, required_skills, nice_to_have, description,
        experience_years, and data_source.
    """
    job_key = job_id.lower().strip().replace(' ', '_')

    if job_key not in JOB_DESCRIPTIONS:
        return {
            'job_id': job_id,
            'found': False,
            'message': f"Job '{job_id}' not found. Available jobs: {', '.join(JOB_DESCRIPTIONS.keys())}.",
            'data_source': 'local_json',
        }

    job = JOB_DESCRIPTIONS[job_key]

    return {
        'job_id': job_key,
        'found': True,
        'title': job['title'],
        'description': job['description'],
        'required_skills': job['required_skills'],
        'nice_to_have': job['nice_to_have'],
        'experience_years': job['experience_years'],
        'data_source': 'local_json',
    }


@mcp.tool(tags={'career', 'resume'})
def get_resume_profile(candidate_id: str) -> dict:
    """
    Get a candidate resume profile by candidate ID.

    Args:
        candidate_id: One of: candidate_1, candidate_2

    Returns:
        Dict with candidate_id, name, current_role, skills, experience,
        learning_goals, and data_source.
    """
    candidate_key = candidate_id.lower().strip().replace(' ', '_')

    if candidate_key not in SAMPLE_RESUMES:
        return {
            'candidate_id': candidate_id,
            'found': False,
            'message': f"Candidate '{candidate_id}' not found. Available candidates: {', '.join(SAMPLE_RESUMES.keys())}.",
            'data_source': 'local_json',
        }

    profile = SAMPLE_RESUMES[candidate_key]

    return {
        'candidate_id': candidate_key,
        'found': True,
        'name': profile['name'],
        'current_role': profile['current_role'],
        'skills': profile['skills'],
        'experience_years': profile['experience_years'],
        'experience': profile['experience'],
        'learning_goals': profile['learning_goals'],
        'data_source': 'local_json',
    }


@mcp.tool(tags={'career', 'gap'})
def get_skill_gap_analysis(job_id: str, candidate_id: str) -> dict:
    """
    Compute a skill gap analysis between a job description and a candidate resume.

    Args:
        job_id: One of: ai_engineer, data_scientist, backend_engineer
        candidate_id: One of: candidate_1, candidate_2

    Returns:
        Dict with matched_skills, missing_required_skills, missing_nice_to_have,
        recommended_learning_topics, and data_source.
    """
    job_key       = job_id.lower().strip().replace(' ', '_')
    candidate_key = candidate_id.lower().strip().replace(' ', '_')

    if job_key not in JOB_DESCRIPTIONS:
        return {
            'found': False,
            'message': f"Job '{job_id}' not found. Available: {', '.join(JOB_DESCRIPTIONS.keys())}.",
            'data_source': 'local_json',
        }
    if candidate_key not in SAMPLE_RESUMES:
        return {
            'found': False,
            'message': f"Candidate '{candidate_id}' not found. Available: {', '.join(SAMPLE_RESUMES.keys())}.",
            'data_source': 'local_json',
        }

    job       = JOB_DESCRIPTIONS[job_key]
    candidate = SAMPLE_RESUMES[candidate_key]

    candidate_skills = set(candidate['skills'])
    required_skills  = set(job['required_skills'])
    nice_to_have     = set(job['nice_to_have'])

    matched_required = sorted(required_skills & candidate_skills)
    missing_required = sorted(required_skills - candidate_skills)
    missing_nice     = sorted(nice_to_have - candidate_skills)

    recommended_topics = set()
    for skill in missing_required + missing_nice:
        skill_info = SKILL_MAP.get(skill, {})
        topic = skill_info.get('learning_topic')
        if topic and topic in VALID_TOPICS:
            recommended_topics.add(topic)

    priority_topics = set()
    for skill in missing_required:
        skill_info = SKILL_MAP.get(skill, {})
        topic = skill_info.get('learning_topic')
        if topic and topic in VALID_TOPICS:
            priority_topics.add(topic)

    return {
        'job_id': job_key,
        'candidate_id': candidate_key,
        'job_title': job['title'],
        'candidate_name': candidate['name'],
        'found': True,
        'matched_skills': matched_required,
        'missing_required_skills': missing_required,
        'missing_nice_to_have': missing_nice,
        'recommended_learning_topics': sorted(priority_topics) + sorted(recommended_topics - priority_topics),
        'data_source': 'local_json',
    }


if __name__ == '__main__':
    mcp.run(transport='http', host='0.0.0.0', port=8004, path='/mcp', log_level='info')


In [ ]:
# Restart MCP server so it picks up the file you just wrote
if 'MCP Server' in _services:
    proc, log, _ = _services.pop('MCP Server')
    proc.terminate()
    log.close()
    time.sleep(1)

start_service('MCP Server', 'mcp.fastmcp_server', 8004)
time.sleep(3)
wait_for_service('http://127.0.0.1:8004/', 'MCP Server')


In [ ]:
%%writefile a2a_agents/remote_agents/assessment_agent/agent_card.py
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentInterface,
    AgentSkill,
)

# The tags below are used by the Orchestrator to route messages.
# Tags like 'quiz', 'assessment', 'score', 'level' ensure that messages
# such as "Quiz me on MCP" or "I got 3 out of 4" are routed to this agent.
assess_level_skill = AgentSkill(
    id='assess_level',
    name='Assess Topic Level Skill',
    description=(
        'Generates assessment questions for a topic, evaluates quiz answers, '
        'and updates the user learning level. Supports follow-up answers within '
        'the same conversation.'
    ),
    tags=['assessment', 'quiz', 'level', 'score', 'questions', 'evaluate', 'test'],
    examples=[
        'Give me a short quiz for MCP.',
        'Assess my level in A2A.',
        'I got 3 out of 4 correct.',
        'What is my current MCP level?',
        'Test my knowledge of RAG.',
        'Quiz me on prompt engineering at intermediate level.',
    ],
)

public_agent_card = AgentCard(
    name='Assessment Agent',
    description=(
        'An agent that assesses user knowledge on topics like MCP, A2A, RAG, '
        'prompt engineering, and Python async. It generates quiz questions, '
        'evaluates results, and updates the user level (beginner -> intermediate -> advanced).'
    ),
    url='http://localhost:8092/',
    supported_interfaces=[AgentInterface(url='http://localhost:8092/', transport='JSONRPC')],
    version='1.0.0',
    capabilities=AgentCapabilities(streaming=True),
    skills=[assess_level_skill],
    default_input_modes=['text'],
    default_output_modes=['text'],
)


### `agent_logic.py` — Your main task

Find the **three TODO blocks** inside `stream()` and fill them in.

> **Why `full_context`?** The agent concatenates conversation history with the current input into a single string before calling `_parse_topic()`. This is how multi-turn memory works: if Turn 1 said *"I want to learn MCP"* and Turn 2 says *"Assess me"*, the topic `mcp` is found in `full_context` even though Turn 2 alone contains no topic.

**MCP tool reference:**

```python
# Flow 1 — update after quiz score:
update = await self._call_tool_json('update_learning_state', {
    'user_id': 'user_1', 'topic': topic,
    'correct_count': correct_count, 'total_count': total_count
})
# returns: {'previous_level': 'beginner', 'new_level': 'intermediate', 'score': 0.75, 'message': '...'}

# Flow 2 — get current level:
state = await self._call_tool_json('get_learning_state', {'user_id': 'user_1', 'topic': topic})
# returns: {'current_level': 'beginner' | 'intermediate' | 'advanced', ...}

# Flow 3 — get quiz questions:
questions = await self._call_tool_json('get_assessment_questions_by_topic', {
    'topic': topic, 'level': level, 'limit': 4
})
# returns: {'questions': [{'question': '...', 'expected_answer': '...'}, ...]}
```

**Yield format** — use this exact dict structure for all three flows:
```python
yield {
    'completed': True,
    'failed': False,
    'input_required': False,
    'content': '<your response string here>'   # ← this is what the user sees
}
```


In [ ]:
%%writefile a2a_agents/remote_agents/assessment_agent/agent_logic.py
import json
import os
import re
from typing import AsyncGenerator

import dotenv
from fastmcp import Client

dotenv.load_dotenv()

KNOWN_TOPICS = ["mcp", "a2a", "rag", "prompt_engineering", "python_async"]
TOPIC_ALIASES = {
    "model context protocol": "mcp",
    "agent to agent": "a2a",
    "agent-to-agent": "a2a",
    "retrieval augmented generation": "rag",
    "retrieval-augmented generation": "rag",
    "prompt": "prompt_engineering",
    "prompting": "prompt_engineering",
    "async": "python_async",
    "asyncio": "python_async",
    "asynchronous": "python_async",
}


def _extract_part_text(part) -> str:
    text = getattr(part, "text", None)
    if not text and hasattr(part, "root"):
        text = getattr(part.root, "text", None)
    if not text and hasattr(part, "model_dump"):
        dumped = part.model_dump()
        if isinstance(dumped, dict):
            text = dumped.get("text")
    return str(text or "")


def _parse_topic(text: str) -> str:
    lower = text.lower()
    for alias, topic in TOPIC_ALIASES.items():
        if alias in lower:
            return topic
    for topic in KNOWN_TOPICS:
        if topic in lower or topic.replace("_", " ") in lower:
            return topic
    return "mcp"


def _parse_score(text: str) -> tuple[int, int] | None:
    patterns = [
        r"(\d+)\s*out of\s*(\d+)",
        r"(\d+)\s*/\s*(\d+)",
        r"(\d+)\s*from\s*(\d+)",
    ]
    lower = text.lower()
    for pattern in patterns:
        match = re.search(pattern, lower)
        if match:
            return int(match.group(1)), int(match.group(2))
    return None


class AgentLogic:
    def __init__(self):
        self.mcp_url = os.getenv("MCP_SERVER_URL", "http://127.0.0.1:8004/mcp")
        self.mcp_client = Client(self.mcp_url)

    async def _call_tool_json(self, tool_name: str, args: dict) -> dict:
        async with self.mcp_client:
            result = await self.mcp_client.call_tool(tool_name, args)
            raw = result.content[0].text if result.content else "{}"
        return json.loads(raw)

    async def stream(self, user_input: str, history=None) -> AsyncGenerator:
        try:
            history_texts = []
            for msg in (history or []):
                for part in msg.parts:
                    text = _extract_part_text(part)
                    if text:
                        history_texts.append(text)

            full_context = " ".join(history_texts + [user_input])
            topic = _parse_topic(full_context)

            score = _parse_score(user_input)
            if score:
                correct_count, total_count = score
                update = await self._call_tool_json(
                    "update_learning_state",
                    {
                        "user_id": "user_1",
                        "topic": topic,
                        "correct_count": correct_count,
                        "total_count": total_count,
                    },
                )
                content = (
                    f"Your {topic} level update:\n"
                    f"- Previous level: {update.get('previous_level')}\n"
                    f"- New level: {update.get('new_level')}\n"
                    f"- Score: {correct_count}/{total_count}\n"
                    f"- {update.get('message', '')}"
                )
                yield {"completed": True, "failed": False, "input_required": False, "content": content}
                return

            lower_input = user_input.lower()
            is_level_query = (
                "what is my level" in lower_input
                or "current level" in lower_input
                or ("my current" in lower_input and "level" in lower_input)
                or "what level" in lower_input
            )
            if is_level_query:
                state = await self._call_tool_json("get_learning_state", {"user_id": "user_1", "topic": topic})
                content = f"Your current {topic} level is {state.get('current_level', 'beginner')}."
                yield {"completed": True, "failed": False, "input_required": False, "content": content}
                return

            state = await self._call_tool_json("get_learning_state", {"user_id": "user_1", "topic": topic})
            level = state.get("current_level", "beginner")
            questions = await self._call_tool_json(
                "get_assessment_questions_by_topic",
                {"topic": topic, "level": level, "limit": 4},
            )

            rows = [f"Assessment questions for {topic} ({level} level):"]
            for idx, q in enumerate(questions.get("questions", []), 1):
                rows.append(f"Question {idx}: {q.get('question', '')}")
            if len(rows) == 1:
                rows.append("No question available for this topic right now.")

            yield {"completed": True, "failed": False, "input_required": False, "content": "\n".join(rows)}

        except Exception as e:
            yield {"completed": False, "failed": True, "input_required": False, "content": f"Error in assessment_agent: {e}"}


In [ ]:
%%writefile a2a_agents/remote_agents/assessment_agent/agent_executor.py
from a2a_agents.base_executor import BaseAgentExecutor
from .agent_logic import AgentLogic


class AgentExecutor(BaseAgentExecutor):
    def _make_logic(self):
        return AgentLogic()


In [ ]:
%%writefile a2a_agents/remote_agents/assessment_agent/__main__.py
from a2a_agents.server_factory import run
from .agent_card import public_agent_card
from .agent_executor import AgentExecutor

if __name__ == '__main__':
    run(public_agent_card, AgentExecutor, port=8092)


---
### Part 3 Checkpoint

After running the four `%%writefile` cells above, these files exist in `a2a_agents/remote_agents/assessment_agent/`:
- `agent_card.py` — skills, tags, and routing metadata
- `agent_logic.py` — your implementation of the three flows
- `agent_executor.py` — A2A executor wiring
- `__main__.py` — uvicorn entry point

Proceed to Part 4 to start the agent and test your implementation.

---

---
## Part 4 — Run Your Assessment Agent

Your agent starts on port 8092. The cell below smoke-tests all three flows directly against your agent (bypassing the Orchestrator).

**What you should see:**
- **Flow 3** (default quiz): a numbered list starting with `"Assessment questions for mcp (beginner level):"`
- **Flow 2** (level query): `"Your current mcp level is beginner."`
- **Flow 1** (score report): a level update message with previous level, new level, and score percentage

If the agent fails to start, run `show_log('Assessment Agent')` in a new cell to see the error output.


In [ ]:
if 'Assessment Agent' in _services:
    proc, log, _ = _services.pop('Assessment Agent')
    proc.terminate()
    log.close()
    time.sleep(1)

print('Starting Assessment Agent...')
start_service('Assessment Agent', 'a2a_agents.remote_agents.assessment_agent', 8092)
time.sleep(4)
ok = wait_for_service('http://localhost:8092/.well-known/agent-card.json', 'Assessment Agent')

if not ok:
    print('\nAgent failed to start. Check the log below:')
    show_log('Assessment Agent')


In [ ]:
# Smoke-test all three flows directly against your Assessment Agent
from tests.helpers import call_agent

print('--- Flow 3: Generate quiz questions (default) ---')
print('Expected: numbered question list for MCP at beginner level')
resp = await call_agent('http://localhost:8092', 'Quiz me on MCP.')
print('\n---Response:---')
print(resp)

print('\n--- Flow 2: Level query ---')
print('Expected: "Your current mcp level is beginner."')
resp = await call_agent('http://localhost:8092', 'What is my current MCP level?')
print('\n---Response:---')
print(resp)

print('\n--- Flow 1: Score reporting ---')
print('Expected: level update showing previous level, new level, and score')
resp = await call_agent('http://localhost:8092', 'I got 3 out of 4 correct on MCP.')
print('\n---Response:---')
print(resp)

---
### Part 4 Checkpoint

After the smoke tests you should have seen:
- **Flow 3**: a numbered list beginning with `"Assessment questions for mcp (beginner level):"`
- **Flow 2**: `"Your current mcp level is beginner."`
- **Flow 1**: a message containing previous level, new level, and a score percentage

**If a flow returned an error**, re-open the `agent_logic` cell, fix your TODO implementation, re-run the `%%writefile` cell, then restart the agent and retry.

---

---
## Part 5 — Orchestrator

The **Orchestrator** (port 8090) routes each user message to the right agent using a keyword-based index built from all agent cards.

**How routing works:**
1. At startup, fetch all agent cards from `agents_registry.json`
2. Build a keyword index from each agent's skill names, descriptions, tags, and examples
3. For each user message, score every agent by counting keyword overlaps
4. Forward the message to any agent scoring within **50% of the top score** — so if Topic Explainer scores 10 and Assessment Agent scores 7, both receive the message; if Assessment Agent scores 3, only Topic Explainer does

**Expected routing:**
- *"Explain MCP"* → **Topic Explainer** (tags: `explain`, `topic`, `concepts`)
- *"Quiz me on A2A"* → **Assessment Agent** (tags: `quiz`, `assessment`, `level`)
- *"Build me a study plan"* → **Study Plan Agent** (tags: `study`, `plan`)

> **Note:** Because routing uses keyword overlap (not semantic similarity), the tags in your Assessment Agent's card directly determine which messages it receives. The card you wrote in Part 3 already includes `quiz`, `assessment`, `score`, and `level`.


In [ ]:
%%html
<div id="routing-diagram" class="mermaid" style="max-width:680px;margin:16px auto">
flowchart LR
    Cards["Agent Cards\n(fetched at startup)"] --> Index["Keyword Index\nper agent"]
    Msg(["User message"]) --> Score["Score each agent\nby keyword overlap"]
    Index --> Score
    Score --> Threshold{"other agents\nwithin 50%\nof top score?"}
    Threshold -->|"Yes"| Multi["Route to\nmultiple agents\n(combine responses)"]
    Threshold -->|"No"| Single["Route to\nbest-match agent only"]
    Multi --> Resp(["Response to user"])
    Single --> Resp
</div>
<script>
  (function () {
    var s = document.createElement("script");
    s.src = "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js";
    s.onload = function () {
      mermaid.initialize({ startOnLoad: false, theme: "neutral" });
      mermaid.run();
    };
    document.head.appendChild(s);
  })();
</script>

In [ ]:
print('Starting Orchestrator...')
start_service('Orchestrator', 'a2a_agents.orchestrator_agent', 8090)
time.sleep(5)
wait_for_service('http://localhost:8090/.well-known/agent-card.json', 'Orchestrator')


In [ ]:
# Raw Orchestrator response: same prompt used in Part 2, now routed automatically
from tests.helpers import call_agent

ORCH = 'http://localhost:8090'
prompt = 'Explain MCP for a beginner'

print('=== Orchestrator Call (with keyword routing) ===')
print(f'Request: "{prompt}"\n')

resp = await call_agent(ORCH, prompt)
print('--- Orchestrator response ---')
print(resp[:600])
if len(resp) > 600:
    print('...(truncated)')

print()
print('--- What happened ---')
print('  1. Orchestrator scored all three agent cards by keyword overlap')
print("  2. 'explain' + 'mcp' + 'beginner' matched Topic Explainer's tags")
print('  3. Orchestrator forwarded the message and relayed Topic Explainer\'s response')
print()
print('  Compare with the direct agent call in Part 2 — the content is identical.')
print('  The Orchestrator adds routing transparency, not content transformation.')

In [ ]:
# Test that the Orchestrator routes correctly to all three agents
from tests.helpers import call_agent

ORCH = 'http://localhost:8090'

print('--- Routing to Topic Explainer ---')
print('Expected: structured MCP explanation (same as direct agent call in Part 2)')
resp = await call_agent(ORCH, 'Explain MCP for a beginner.')
print(resp[:400], '...' if len(resp) > 400 else '')

print('\n--- Routing to Assessment Agent ---')
print('Expected: quiz questions for A2A at your current level')
resp = await call_agent(ORCH, 'Quiz me on A2A.')
print(resp[:400], '...' if len(resp) > 400 else '')

print('\n--- Routing to Study Plan Agent ---')
print('Expected: ordered study steps for RAG with time estimates')
resp = await call_agent(ORCH, 'Build me a 2-hour study plan for RAG.')
print(resp[:400], '...' if len(resp) > 400 else '')


---
### Part 5 Checkpoint

After the routing tests you should have seen three distinct responses:
- Topic Explainer received *"Explain MCP"* and returned a structured concept explanation
- Assessment Agent received *"Quiz me on A2A"* and returned numbered quiz questions
- Study Plan Agent received *"Build me a study plan"* and returned ordered study steps

**If a message was routed to the wrong agent**, check that your Assessment Agent's `agent_card.py` tags include words like `quiz`, `assessment`, `score`, and `level`. Re-run the card cell and restart the Assessment Agent.

---

---
## Part 6 — Full Test Suite

Run the automated tests to validate the whole system. The tests follow a **bottom-up pyramid** — start at the base and work up:

```
test_memory.py    ← multi-turn conversation with shared context_id  (all services)
test_e2e.py       ← end-to-end flows via Orchestrator               (all services)
test_agents.py    ← conversation scenarios per agent                 (agents + MCP)
test_mcp.py       ← all 8 MCP tools, happy paths + error cases      (MCP alone) ← start here
```

> **Note:** The `tests/` folder also contains `test_mcp.py` and `test_agents.py` — MCP tools and individual agents were already validated earlier in this notebook. Here we run only the integration tests.

| Test file | What it covers |
|-----------|----------------|
| `test_e2e.py` | End-to-end flows via Orchestrator + offline guarantees |
| `test_memory.py` | 4-turn multi-turn conversation with shared context_id |


In [ ]:
# End-to-end tests via the Orchestrator
!python -m pytest tests/test_e2e.py -v --tb=short 2>&1 | head -100


In [ ]:
%%html
<div id="memory-diagram" class="mermaid" style="max-width:760px;margin:16px auto">
sequenceDiagram
    participant Client
    participant Orch as Orchestrator :8090
    participant Agent as Assessment Agent :8092

    Client->>Orch: Turn 1 — context_id=abc<br/>"I want to learn MCP. I am a beginner."
    Orch->>Agent: context_id=abc, history=[]
    Agent-->>Client: "MCP is a protocol..."

    Client->>Orch: Turn 2 — context_id=abc<br/>"Assess me."
    Note over Orch: history = [Turn 1]
    Orch->>Agent: context_id=abc, history=[Turn 1]
    Note over Agent: full_context includes Turn 1<br/>_parse_topic finds "mcp"
    Agent-->>Client: "Assessment questions for mcp..."

    Client->>Orch: Turn 3 — context_id=abc<br/>"I got 3 out of 4 correct."
    Note over Orch: history = [Turn 1, Turn 2]
    Orch->>Agent: context_id=abc, history=[Turn 1, Turn 2]
    Note over Agent: full_context includes "mcp"<br/>_parse_score finds 3/4
    Agent-->>Client: "Level updated: beginner → intermediate"

    Client->>Orch: Turn 4 — context_id=abc<br/>"Build me a 2-hour study plan."
    Note over Orch: history = [Turn 1, Turn 2, Turn 3]
    Orch->>Agent: forwarded to Study Plan Agent with history
    Agent-->>Client: "Study plan for mcp at intermediate level..."
</div>
<script>
  (function () {
    var s = document.createElement("script");
    s.src = "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js";
    s.onload = function () {
      mermaid.initialize({ startOnLoad: false, theme: "neutral" });
      mermaid.run();
    };
    document.head.appendChild(s);
  })();
</script>

In [ ]:
# Multi-turn memory test — pytest pass/fail
!python -m pytest tests/test_memory.py -v --tb=short 2>&1 | head -60

# Live 4-turn conversation — watch how context_id threads topic and level across turns:
# Turn 1: "I want to learn MCP. I am a complete beginner."   → establishes topic=MCP
# Turn 2: "Assess me on MCP."                                → topic resolved from history
# Turn 3: "I got 3 out of 4 correct."                       → topic + score resolved from history
# Turn 4: "Build me a 2-hour study plan based on my current level." → uses updated MCP level
!python tests/test_memory.py


In [ ]:
# Run this cell to stop all background services cleanly
# kill_all_services()
